In [ ]:
!pip install Biopython # Si no se tiene la librería Biopython

from Bio import Entrez
Entrez.email = "user@gmail.com" # Siempre hay que identificarse 

#Primero es importante saber: qué podemos buscar con Entrez en la base de datos nucleotide? (Se puede aplicar a todas las db)

stream = Entrez.einfo (db="assembly")
record = Entrez.read(stream)

for field in record["DbInfo"]["FieldList"]:
    print("%(Name)s, %(FullName)s, %(Description)s" % field)

ALL, All Fields, All terms from all searchable fields
UID, UID, Unique number assigned to publication
FILT, Filter, Limits the records
ACCN, Accession, Chromosome accessions
ASAC, Assembly Accession, Space delimited assembly accessions w/ & w/o versions
ASLV, Assembly Level, How assembled is this assembly. 'Contig' to 'Chromosome'
TXID, Taxonomy ID, Taxonomy ID
ORGN, Organism, Exploded organism names
RUID, RefSeq ID, Id of RefSeq Assembly.
GUID, GenBank ID, Id of GenBank synonym of this Assembly.
UIDS, All Uids, Pair-id, GB-id, and RS-id of this Assembly.
PROJ, BioProject IDs and Accessions, Uid and accessions of this assembly's projects
SAMP, BioSample, BioSample Accession and Id
NAME, Assembly Name, Assembly name
ALLN, All Names, All names, space separated
DESC, Description, Assembly description
COV, Coverage, Sequencing coverage
TYPE, Assembly Type, Type of the assembly
SRDT, Date - Sequences Release, Date the most recent sequence went live in ID
UPDT, Date - Assembly Update, Date t

In [ ]:
from Bio import Entrez

# Ahora que ya sabemos los campos buscaremos secuencias de blaZ en S. aureus de manera específica

stream = Entrez.esearch(db="nucleotide", 
                        term="blaZ[Gene] AND Staphylococcus aureus[Organism] AND complete cds [Title]")
record = Entrez.read(stream) #Es mejor Entrez.read que no handle.read, porque esta es la funcion de parsing 
# de Entrez, y lo que hace es convertirlo en un diccionario que pueda interpretar Python

stream.close()

print(f"Secuencias encontradas: {record['Count']}")

Secuencias encontradas: 29


In [ ]:
#Ahora que sabemos lo que hay, lo descargaremos con el módulo EFetch de Entrez
from Bio import Entrez  

Entrez.email = "user@gmail.com"  

#Todos los parámetros están  en https://www.ncbi.nlm.nih.gov/books/NBK25499/#chapter4.EFetch

stream = Entrez.efetch(db="nucleotide",
                       id=record['IdList'],
                       rettype="fasta",
                       retmode="text")

# The open() function opens a file, and returns it as a file object. Si este archivo no existe lo crea!
# La estructura es open("file", mode). Como modo hemos puesto w = write, para que escriba lo que sea que le hemos dicho debajo
# https://www.w3schools.com/python/ref_func_open.asp
 
with open("blaZ_sequences.fasta", "w") as output_file:
    output_file.write(stream.read())
stream.close()

#Esto nos ha creado un fichero llamado "blaZ_sequences" donde estan todas las secuencias que se ajustan a la búsqueda anterior que hemos hecho

In [ ]:
# Si vemos el fichero, nos damos cuenta que no todas las secuencias se ajustan solo a blaZ. Tenemos que limpiarlas un poco
# Para definir el rango del filtro primero tendremos que ver una pequeña descripcion de nuestras sec

from Bio import SeqIO # Libreria para leer, escribir y manipular archivos de secuencias biológicas (FASTA, GenBank)

for secuencia in SeqIO.parse("blaZ_sequences.fasta", "fasta"):
    print(f"{secuencia.id}, {len(secuencia)} pb, {secuencia.description}")

# Al correr esto vemos que aprox. las que estan entre 900 - 1350 bp (U58139.2) son b-lactam solas, pero las que superan este numero
# contienen otros elementos. Estas secuencias no las queremos.

MW453507.1, 904 pb, MW453507.1 Staphylococcus aureus strain YCH5 penicillin-hydrolyzing class A beta-lactamase (blaZ) gene, complete cds
MW453506.1, 904 pb, MW453506.1 Staphylococcus aureus strain YCH71 penicillin-hydrolyzing class A beta-lactamase (blaZ) gene, complete cds
MW453505.1, 904 pb, MW453505.1 Staphylococcus aureus strain YCH196 penicillin-hydrolyzing class A beta-lactamase (blaZ) gene, complete cds
MW453504.1, 904 pb, MW453504.1 Staphylococcus aureus strain YCH225 penicillin-hydrolyzing class A beta-lactamase (blaZ) gene, complete cds
MW453503.1, 904 pb, MW453503.1 Staphylococcus aureus strain YCH179 penicillin-hydrolyzing class A beta-lactamase (blaZ) gene, complete cds
MW453502.1, 904 pb, MW453502.1 Staphylococcus aureus strain YCH32 penicillin-hydrolyzing class A beta-lactamase (blaZ) gene, complete cds
MW453501.1, 904 pb, MW453501.1 Staphylococcus aureus strain YCH9 penicillin-hydrolyzing class A beta-lactamase (blaZ) gene, complete cds
MW453500.1, 904 pb, MW453500.1 St

In [ ]:
#Sabiendo esto, vamos a crear otro documento con las secuencias filtradas:

# Aqui NO usamos el comando open() porque este es propio de python y no entiende el formato fasta. Tendriamos que escribir mucho código adicional
# Para eso nos sirve SeqIO, y antes no lo hemos usado pq habia que descargar las secuencias para empezar. Las principales diferencias entre los 2:

# open() + efetch: para descargar datos de internet y guardarlos en bruto como texto
# SeqIO: para leer y manipular archivos de secuencias que ya tienes guardados

secuencias_filtradas = [] # Creamos una lista para nuestras secuencias definitivas

for secuencia in SeqIO.parse("blaZ_sequences.fasta", "fasta"):
    if len(secuencia) <= 1400:
        secuencias_filtradas.append(secuencia)

# Las guardamos en otro archivo fasta

SeqIO.write(secuencias_filtradas, "blaZ_filtradas.fasta", "fasta")


22